In [1]:
!pip install transformers torch scikit-learn pandas numpy --quiet

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
from google.colab import files
uploaded = files.upload()

Saving flood_dataset.csv to flood_dataset.csv


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [5]:
df = pd.read_csv("flood_dataset.csv")

print("Original Columns:", df.columns)

df.columns = df.columns.str.strip().str.lower()

print("After cleaning:", df.columns)

df = df.rename(columns={
    "comments": "comment",
    "score": "severity"
})

print("Final Columns:", df.columns)
df.head()

Original Columns: Index(['Comments', 'Score'], dtype='object')
After cleaning: Index(['comments', 'score'], dtype='object')
Final Columns: Index(['comment', 'severity'], dtype='object')


,comment,severity
0,The flood water is slowly rising near our street.,0.32
1,Minor flood in the area but authorities are mo...,0.28
2,The flood has entered our backyard but not the...,0.41
3,Heavy rain might cause a flood by tonight.,0.35
4,Small flood reported near the riverbank.,0.30


In [6]:
print("Original shape:", df.shape)

df = df.drop_duplicates()
df = df.dropna()

df["severity"] = df["severity"].astype(float)
df = df[(df["severity"] >= 0) & (df["severity"] <= 1)]

print("Cleaned shape:", df.shape)

Original shape: (1499, 2)
Cleaned shape: (1454, 2)


In [7]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["comment"].tolist(),
    df["severity"].tolist(),
    test_size=0.2,
    random_state=42
)

print("Train size:", len(train_texts))
print("Test size:", len(test_texts))

Train size: 1163
Test size: 291


In [8]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
class FloodDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FloodDataset(train_encodings, train_labels)
test_dataset = FloodDataset(test_encodings, test_labels)

In [10]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=1
)

model.config.problem_type = "regression"
model.to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)

In [12]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(labels, predictions)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,R2
1,No log,0.020115,0.100935,0.020115,0.141828,0.711220
2,No log,0.015362,0.078196,0.015362,0.123942,0.779465
3,No log,0.014682,0.074897,0.014682,0.121170,0.789217


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=219, training_loss=0.030952640864402737, metrics={'train_runtime': 760.7772, 'train_samples_per_second': 4.586, 'train_steps_per_second': 0.288, 'total_flos': 26177626634094.0, 'train_loss': 0.030952640864402737, 'epoch': 3.0})

In [15]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.014682264998555183, 'eval_MAE': 0.07489686459302902, 'eval_MSE': 0.014682263135910034, 'eval_RMSE': 0.12117038885763318, 'eval_R2': 0.7892169952392578, 'eval_runtime': 13.7419, 'eval_samples_per_second': 21.176, 'eval_steps_per_second': 1.383, 'epoch': 3.0}


In [ ]:
save_path = "./flood_model"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [17]:
def predict_severity(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        score = outputs.logits.squeeze().item()

    score = max(0, min(1, score))
    return score

In [22]:
user_comment = input("Enter a flood-related comment: ")

predicted_score = predict_severity(user_comment)

print("\nPredicted Severity Score:", round(predicted_score, 4))

if predicted_score <= 0.3:
    print("Severity Level: LOW")
elif predicted_score <= 0.7:
    print("Severity Level: MEDIUM")
else:
    print("Severity Level: HIGH")

Enter a flood-related comment: Severe flood has submerged the highway.

Predicted Severity Score: 0.997
Severity Level: HIGH
